# Обучение Tri-View 3D Autoencoder (Этап 1)

Архитектура с skip connections от lifted views + улучшенный training pipeline.

**Запуск:** Google Colab с GPU T4.

In [ ]:
# 1. Монтируем Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Скачиваем актуальный код с GitHub
%cd /content
!rm -rf MultiView_3D_Prediction
!git clone https://github.com/3x6dll9ff/MultiView_3D_Prediction.git
%cd MultiView_3D_Prediction

In [ ]:
# 3. Устанавливаем зависимости
!pip install -r requirements.txt

In [ ]:
# 4. Распаковываем подготовленные данные
!rm -rf /content/data
!mkdir -p /content/data
!unzip -q /content/drive/MyDrive/data_processed.zip -d /content/data
print("Распаковка завершена.")

# Проверяем данные
import os
DATA_PATH = os.popen('find /content/data -name "metadata.csv" | head -n 1 | xargs dirname').read().strip()
print(f"Данные: {DATA_PATH}")
print(f"Сэмплов: {len([f for f in os.listdir(os.path.join(DATA_PATH, 'top_proj')) if f.endswith('.npy')])}")

In [ ]:
# 5. Обучение base autoencoder
# - skip connections от lifted views на каждом уровне decoder
# - AdamW + LR warmup (5 эпох)
# - Аугментация: flip, noise, brightness
# - Composite loss: BCE + Dice + Projection(BCE) + Boundary
!DATA_PATH=$(find /content/data -name "metadata.csv" | head -n 1 | xargs dirname) && \
 echo "Используем данные из: $DATA_PATH" && \
 python3 src/train_reconstruction.py \
    --data_dir "$DATA_PATH" \
    --input_mode tri \
    --epochs 80 \
    --batch_size 8 \
    --lr 1e-3 \
    --num_workers 2 \
    --early_stopping_patience 15

In [ ]:
# 6. Сохраняем результаты на Google Drive
import os, shutil
DRIVE_DIR = '/content/drive/MyDrive/MultiView_Results'
os.makedirs(DRIVE_DIR, exist_ok=True)

for f in ['best_autoencoder.pt', 'checkpoint_best_autoencoder.pt']:
    src = os.path.join('results', f)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(DRIVE_DIR, f))
        print(f"Сохранено: {f}")

metrics_dir = os.path.join(DRIVE_DIR, 'metrics')
os.makedirs(metrics_dir, exist_ok=True)
for f in os.listdir('results/metrics'):
    src = os.path.join('results/metrics', f)
    if os.path.isfile(src):
        shutil.copy2(src, os.path.join(metrics_dir, f))
        print(f"Сохранено: metrics/{f}")

print(f"\nВсе результаты на Drive: {DRIVE_DIR}")